In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
print("All imports successful!")

All imports successful!


In [5]:
# ============================================================
# CELL 1 — Imports & configuration
# Industry practice: all imports at the top, never scattered
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Industry practice: define paths once, never hardcode them inline
RAW_DATA_PATH = Path('../data/raw/superstore.csv')
DB_PATH       = Path('../data/processed/superstore.db')
REPORT_PATH   = Path('../reports/')

print("Libraries loaded. Paths configured.")

Libraries loaded. Paths configured.


In [12]:
# ============================================================
# CELL 2 — Load the data
# ============================================================

RAW_DATA_PATH = r"C:\Users\shokd\Desktop\Superstore\sales-revenue-intelligence\data\train.csv"

df = pd.read_csv(
    RAW_DATA_PATH,
    encoding='latin-1',
    parse_dates=['Order Date', 'Ship Date']
)

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset loaded: 9,800 rows × 18 columns


In [14]:
# ============================================================
# CELL 3 — The professional first-look checklist
# Every analyst runs these 6 checks before touching the data
# ============================================================

print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)

# 1. Shape
print(f"\nRows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

# 2. Column names and data types
print("\nColumn info:")
print(df.dtypes.to_string())

# 3. Missing values — the most critical first check
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)
null_df = pd.DataFrame({'Missing Count': null_counts, 'Missing %': null_pct})
null_df = null_df[null_df['Missing Count'] > 0]

print(f"\nColumns with missing values: {len(null_df)}")
if len(null_df) > 0:
    print(null_df)
else:
    print("None found.")

# 4. Duplicate rows
dupes = df.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")

# 5. Basic statistics for numeric columns
print("\nNumeric summary:")
print(df.describe().round(2).to_string())

# 6. First 3 rows to sanity-check structure
print("\nSample rows:")
print(df.head(3).to_string())

DATASET OVERVIEW

Rows:    9,800
Columns: 18

Column info:
Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code      float64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64

Columns with missing values: 1
             Missing Count  Missing %
Postal Code             11       0.11

Duplicate rows: 0

Numeric summary:
        Row ID  Postal Code     Sales
count  9800.00      9789.00   9800.00
mean   4900.50     55273.32    230.77
std    2829.16     32041.22    626.65
min       1.00      1040.00      0.44
25%    2450.75     23223.00     17.25
50%    4900.50     58103.00     54.49
75%    7350.25     90008.00    210.60
max    9800.00     99301.00  22638.48

Sam

In [ ]:
# ============================================================
# CELL 4 — Data cleaning (professional style)
# Work on a copy, never mutate the raw data
# ============================================================

df_clean = df.copy()

# --- 1. Standardise column names ---
# Spaces in column names cause bugs in SQL and pandas chaining
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)
print("Columns standardised:", df_clean.columns.tolist())

# --- 2. Parse date columns, extract features ---
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])
df_clean['ship_date']  = pd.to_datetime(df_clean['ship_date'])

df_clean['order_year']    = df_clean['order_date'].dt.year
df_clean['order_month']   = df_clean['order_date'].dt.month
df_clean['order_quarter'] = df_clean['order_date'].dt.quarter
df_clean['ship_days']     = (df_clean['ship_date'] - df_clean['order_date']).dt.days

print("\nDate features created.")

# --- 3. Check for business logic violations ---
# A ship date BEFORE the order date is impossible — catch it
invalid_ship = df_clean[df_clean['ship_days'] < 0]
print(f"Orders with negative ship days (data error): {len(invalid_ship)}")

# Negative profit is valid (discounts can cause losses) — but flag it
negative_profit = df_clean[df_clean['profit'] < 0]
print(f"Orders with negative profit: {len(negative_profit)} "
      f"({len(negative_profit)/len(df_clean)*100:.1f}%)")

# --- 4. Create a derived metric: profit margin ---
# This is a KEY business metric — never just use raw profit
df_clean['profit_margin'] = (df_clean['profit'] / df_clean['sales']).round(4)

# --- 5. Drop true duplicates (if any found earlier) ---
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after  = len(df_clean)
print(f"\nRows removed as duplicates: {before - after}")

print(f"\nClean dataset shape: {df_clean.shape}")